In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is seeking information about a fictional location, Lunapolis, including its capital, weather, population, and potential labor issues.\n\n## SUMMARY\n\n- The capital of the moon is identified as Lunapolis.\n- The weather in Lunapolis is reported as clear skies, with a high of 120°C and a low of -100°C.\n- There are approximately 100,000 cheese miners living in Lunapolis.\n- It is indicated that the cheese miners' union is likely to strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='99989c25-aad6-4543-9ee7-c3503b521746'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='6de1a6a2-3f4c-4892-8a48-f6f7ceae689f'),
              

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

The user is seeking information about a fictional location, Lunapolis, including its capital, weather, population, and potential labor issues.

## SUMMARY

- The capital of the moon is identified as Lunapolis.
- The weather in Lunapolis is reported as clear skies, with a high of 120°C and a low of -100°C.
- There are approximately 100,000 cheese miners living in Lunapolis.
- It is indicated that the cheese miners' union is likely to strike due to dissatisfaction with the new president.

## ARTIFACTS

None

## NEXT STEPS

None


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='37d83cb1-eac4-4fdf-a494-11118e9ff5a1'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='607964db-278b-4be1-ae04-a56e7bc2677e', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='c7ca2b0c-81ad-4431-b4bc-721bd868756f'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='a05636ce-3eac-4af3-829d-822f40bfc16c', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='85da174e-1ce4-40f3-a4ff-c6c60cac17c3'),
              AIMessage(content='I can’t measure your device’s temperature from here. If it fee

In [8]:
print(response["messages"][-1].content)

I can’t measure your device’s temperature from here. If it feels hot or you smell something burnt, that’s a sign it needs to cool before you try again.

What to do now:
- If it’s hot, unplug it and let it cool in a well-ventilated area for at least 30 minutes.
- While cooling, don’t try to power it on. Check vents for dust and blockages.
- If you can safely open it (and it’s allowed by warranty), you can gently clean dust from vents with compressed air.
- After it’s cool, try a clean power cycle: unplug, remove the battery if it’s removable, hold the power button for 15–20 seconds, then reconnect power (and reinsert the battery) and try to power on.
- If it’s a laptop, you can try starting with the charger connected but the battery removed (if possible).
- If it’s a desktop, double-check the power cable and PSU switch, and try a minimal boot (RAM only, one device at a time).

If you can tell me:
- What type of device (laptop, desktop, phone/tablet, other)?
- The model or OS?
- Any LED 